In [ ]:
import numpy as np
import pandas as pd

In [ ]:

# ----------------------------------------------------------------------------
# Funções Auxiliares
# ----------------------------------------------------------------------------
def _doy365(date):
    """Dia do ano com 366 -> 365 (ano bissexto), como no cos de YDOY."""
    return np.minimum(date.dt.dayofyear.to_numpy(), 365)


def _learn_fills(train_val, strategy, doy_half_window):
    """Aprende, SÓ em train_val, as medianas usadas na imputação."""
    fill_cols = ["CHL_log", "CHL_uncertainty"]
    obs = train_val.dropna(subset=fill_cols)
    fills = {"global": obs[fill_cols].median()}
    if strategy == "season_median":
        fills["season"] = obs.groupby("Season")[fill_cols].median()
    if strategy == "doy_window_median":
        fills["obs_doy"] = _doy365(obs["Date"])
        fills["obs_vals"] = obs[fill_cols].to_numpy()
        fills["half_window"] = doy_half_window
    return fills


def _apply_fills(d, strategy, fills):
    """Preenche CHL_log e CHL_uncertainty em falta (imputação em log(CHL))."""
    d = d.copy()
    fill_cols = ["CHL_log", "CHL_uncertainty"]
    miss = d[fill_cols].isna().any(axis=1).to_numpy()
    if not miss.any():
        return d

    if strategy == "global_median":
        for c in fill_cols:
            d[c] = d[c].fillna(fills["global"][c])

    elif strategy == "season_median":
        for c in fill_cols:
            by_season = d["Season"].map(fills["season"][c])
            d[c] = d[c].fillna(by_season).fillna(fills["global"][c])

    elif strategy == "doy_window_median":
        doy = _doy365(d["Date"])
        for i in np.flatnonzero(miss):
            diff = np.abs(fills["obs_doy"] - doy[i])
            diff = np.minimum(diff, 365 - diff)           # distância circular
            in_win = diff <= fills["half_window"]
            for j, c in enumerate(fill_cols):
                if np.isnan(d[c].iloc[i]):
                    v = (np.median(fills["obs_vals"][in_win, j])
                         if in_win.any() else fills["global"][c])
                    d.iloc[i, d.columns.get_loc(c)] = v
    return d


# ----------------------------------------------------------------------------
# Função principal
# ----------------------------------------------------------------------------
def prepare_data(strategy, split_test, one_hot_cluster=True, sin_doy=True, test_frac=0.15, doy_half_window=15,
                 seed=42):
    """Prepara os dados do Task 1.

    Parâmetros
    ----------
    strategy : str, estratégia para os missing values (CHL, CHL_uncertainty)
        "drop_rows"         – remove linhas de train/val com missing
        "drop_features"     – remove CHL_log e CHL_uncertainty
        "global_median"     – mediana global (CHL em log)
        "season_median"     – mediana por estação do ano
        "doy_window_median" – mediana numa janela de ±doy_half_window dias
    split_test : str, forma de separar o conjunto de test
        "recent"     – 15% mais recentes, sem linhas com missing
        "stratified" – 15% do total, só linhas completas, espalhadas no tempo
                       e por estações, preservando a mistura de classes
    one_hot_cluster : bool – spatial_cluster -> 6 colunas 0/1
    sin_doy : bool – acrescenta sin(2π·DOY/365)

    Devolve
    -------
    (X_train_val, y_train_val, X_test, y_test)
    """

    # Read data
    df = pd.read_excel("classification_task1_train.xlsx")
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values(["Date", "spatial_cluster"], kind="stable").reset_index(drop=True)


    # Separar os dados de test dos restantes dados
    if split_test == "recent":
        n_test = int(round(test_frac * len(df)))
        cutoff = df["Date"].iloc[len(df) - n_test]
        is_test = (df["Date"] >= cutoff).to_numpy()
        complete = df[["CHL", "CHL_uncertainty"]].notna().all(axis=1).to_numpy()
        test_mask = is_test & complete      # linhas completas do test
        rest_mask = ~is_test    

    else:
        rng = np.random.default_rng(seed)
        d = df.assign(_complete=df[["CHL", "CHL_uncertainty"]].notna().all(axis=1))

        by_date = d.groupby("Date").agg(n_rows=("HAB", "size"),
                                        all_complete=("_complete", "all"),
                                        hab=("HAB", "max"),
                                        season=("Season", "first"))
        elig = by_date[by_date["all_complete"]]

        n_target_dates = int(round(test_frac * len(df) / elig["n_rows"].mean()))
        n_target_dates = min(n_target_dates, len(elig))

        chosen = []
        for _, g in elig.groupby(["season", "hab"]):
            k = int(round(n_target_dates * len(g) / len(elig)))
            k = min(max(k, 1), len(g))                                        # >=1 data por estrato
            pos = ((rng.uniform() + np.arange(k)) * len(g) / k).astype(int)   # posições espaçadas
            chosen.extend(g.index[pos])

        test_mask = df["Date"].isin(chosen).to_numpy()    
        rest_mask = ~test_mask

    train_val, test = df[rest_mask].copy(), df[test_mask].copy()



    # log(CHL), sin_doy, one-hot do cluster
    # (aplicadas igualmente a train_val e test, para as colunas coincidirem) 
    # pode-se fazer isto porque não aprende nada
    train_val = train_val.copy()
    test = test.copy()
    train_val["CHL_log"] = np.log(train_val["CHL"]) 
    test["CHL_log"] = np.log(test["CHL"])                    
    if sin_doy:
        train_val["sin_doy"] = np.sin(2 * np.pi * _doy365(train_val["Date"]) / 365)
        test["sin_doy"] = np.sin(2 * np.pi * _doy365(test["Date"]) / 365)
    if one_hot_cluster:
        for c in range(6):                       
            train_val[f"cluster_{c}"] = (train_val["spatial_cluster"] == c).astype(int)
            test[f"cluster_{c}"] = (test["spatial_cluster"] == c).astype(int)


    # Estratégia para missing values (só para train_val, porque o test já é completo)
    if strategy == "drop_rows":
        train_val = train_val.dropna(subset=["CHL_log", "CHL_uncertainty"])
    elif strategy != "drop_features":
        fills = _learn_fills(train_val, strategy, doy_half_window)
        train_val = _apply_fills(train_val, strategy, fills)

    # Colunas finais (as restantes são redundantes e são descartadas)
    cols = ["Season", "SST", "UI", "YDOY"]
    if sin_doy:
        cols.append("sin_doy")
    if strategy != "drop_features":
        cols += ["CHL_log", "CHL_uncertainty"]
    cols += ([f"cluster_{c}" for c in range(6)]
             if one_hot_cluster else ["spatial_cluster"])

    assert not test[cols].drop(columns="Season").isna().any().any(), "test com NaN"
    assert not train_val[cols].isna().any().any(), "train_val com NaN"

    return (train_val[cols], train_val["HAB"],
            test[cols], test["HAB"])

# Data split, preprocessing and missing values treatment
Este notebook:
1. lê os dados e faz o **split cronológico** em train / validation / test (70 / 15 / 15 em linhas), removendo depois as linhas incompletas do test para que o **test não tenha missing values**;
2. define o **pré-processamento** (log da CHL, features sazonais, one-hot do cluster, standardização) com parâmetros aprendidos **apenas no train** e aplicados ao train e à validation (o test fica intocado);
3. implementa 5 estratégias para os missing values de `CHL` e `CHL_uncertainty`:
- `drop_rows`, 
- `drop_features`,
- `global_median`,
- `season_median`,
- `doy_window_median`.

In [3]:
import numpy as np
import pandas as pd

# Estratégias disponíveis
STRATEGIES = ["drop_rows", "drop_features", "global_median", "season_median", "doy_window_median"]

# Semi-largura da janela (em dias do ano) usada em "doy_window_median"
DOY_HALF_WINDOW = 15

# Se True, a validation usada para comparar estratégias contém apenas linhas completas.
# Assim, todas as estratégias são avaliadas exatamente nas mesmas linhas e a validation
# imita o test (que é sempre completo).
VAL_COMPLETE_ONLY = False

## Read data and split train / validation / test

**Critério do split (cronológico, sem partir datas ao meio):**

1. Ordenam-se as linhas por data e divide-se em **train / validation / test** com as proporções `P_TRAIN / P_VAL / P_TEST` (70 / 15 / 15), usando fronteiras em datas inteiras (a mesma data pode aparecer em 2 clusters e nunca fica em dois conjuntos).
2. No **test** removem-se as linhas incompletas (as removidas ficam em `discarded`, não são usadas em mais nada), de modo a que o test não tenha missing values.

O train e a validation mantêm os missing values.

In [195]:
# Read data
df = pd.read_excel("classification_task1_train.xlsx")

df["Date"] = pd.to_datetime(df["Date"])


# acrescenta a coluna "complete" com TRUE se a linha estiver completa, e FALSE se estiver incompleta
df["complete"] = df[["CHL", "CHL_uncertainty"]].notna().all(axis=1) 

In [196]:
# Separar em dados de train, validation e test:

cum_rows = df.groupby("Date").size().cumsum()                      # índice das datas (Atenção: começa no 2)
N = len(df)                                                        # nº total de linhas


P_TRAIN, P_VAL, P_TEST = 0.70, 0.15, 0.15       # Proporções pretendidas             

train_end = cum_rows[cum_rows >= round(P_TRAIN * N)].index[0]               # última data do train
val_end = cum_rows[cum_rows >= round((P_TRAIN + P_VAL) * N)].index[0]       # última data da validation



train = df[df["Date"] <= train_end].copy()
val = df[(df["Date"] > train_end) & (df["Date"] <= val_end)].copy()
test_all = df[df["Date"] > val_end]



test       = test_all[test_all["complete"]].copy()                 # só linhas completas de test
discarded  = test_all[~test_all["complete"]].copy()                # incompletas do período de test

'''# Verificações de segurança
assert test["complete"].all(), "O test tem missing values!"
assert train["Date"].max() < val["Date"].min() <= val["Date"].max() < test["Date"].min()
assert len(train) + len(val) + len(test) + len(discarded) == N'''

'# Verificações de segurança\nassert test["complete"].all(), "O test tem missing values!"\nassert train["Date"].max() < val["Date"].min() <= val["Date"].max() < test["Date"].min()\nassert len(train) + len(val) + len(test) + len(discarded) == N'

In [197]:
def split_summary(name, d):
    return {
        "split": name, "n": len(d), "% de N": round(100 * len(d) / N, 1),
        "início": d["Date"].min().date() if len(d) else None,
        "fim": d["Date"].max().date() if len(d) else None,
        "HAB=1": int(d["HAB"].sum()) if len(d) else 0,
        "% HAB": round(100 * d["HAB"].mean(), 1) if len(d) else np.nan,
        "% CHL em falta": round(100 * (~d["complete"]).mean(), 1) if len(d) else np.nan,
    }

pd.DataFrame([
    split_summary("train", train),
    split_summary("validation", val),
    split_summary("test", test),
    split_summary("descartadas (incompletas no período de test)", discarded),
]).set_index("split")

,n,% de N,início,fim,HAB=1,% HAB,% CHL em falta
split,,,,,,,
train,744,70.0,2013-02-19,2019-01-02,78,10.5,47.7
validation,161,15.1,2019-01-07,2019-12-30,29,18.0,46.0
test,87,8.2,2020-01-20,2020-12-29,21,24.1,0.0
descartadas (incompletas no período de test),71,6.7,2020-01-13,2020-12-28,10,14.1,100.0


In [198]:
# Validation usada para comparar estratégias
val_eval = val[val["complete"]].copy() if VAL_COMPLETE_ONLY else val.copy()
print(f"validation: {len(val)} linhas -> usadas na avaliação: {len(val_eval)} | HAB=1: {int(val_eval['HAB'].sum())}")

validation: 161 linhas -> usadas na avaliação: 161 | HAB=1: 29


## 2. Pré-processamento e imputação

Fluxo para cada estratégia (**tudo aprendido só no train**):

1. `add_base_features`: log da CHL, `sin_doy` (complementa o `YDOY`, que é só um cosseno) e dia do ano.
2. Imputação (ou remoção de linhas/colunas) conforme a estratégia.
4. Standardização (média 0, desvio 1) das variáveis contínuas, com média/desvio do **train**.

A imputação é feita **em log(CHL)** (a CHL é muito assimétrica). A `CHL_uncertainty` é usada na escala original.
`Season`, `Day`, `Month` e `Year` não entram como features (a informação sazonal está em `YDOY`/`sin_doy`).

In [199]:
# Nomes internos das colunas com missing (depois do log da CHL)
MISS = ["CHL_log", "CHL_uncertainty"]
BASE_NUM  = ["SST", "UI", "YDOY", "sin_doy"]

splits = {"train": train, "val": val_eval, "test": test}


def add_base_features(d):
    """Cria CHL_log, doy e sin_doy"""
    out = pd.DataFrame(index=d.index)
    out["Date"] = d["Date"]
    out["Season"] = d["Season"]
    out["spatial_cluster"] = d["spatial_cluster"]
    out["SST"] = d["SST"]
    out["UI"] = d["UI"]
    out["YDOY"] = d["YDOY"]
    out["CHL_uncertainty"] = d["CHL_uncertainty"]
    out["CHL_log"] = np.log(d["CHL"])               # NaN mantém-se NaN
    doy = d["Date"].dt.dayofyear.clip(upper=365)
    out["doy"] = doy
    out["sin_doy"] = np.sin(2 * np.pi * doy / 365)
    if "HAB" in d.columns:
        out["HAB"] = d["HAB"]
    return out

In [200]:
def add_base_features(d):
    """Pré-processamento sem aprendizagem: log da CHL, dia do ano e sin(dia do ano)."""
    doy = d["Date"].dt.dayofyear.clip(upper=365)                      # 366 -> 365
    return d.assign(CHL_log=np.log(d["CHL"]), doy=doy, sin_doy=np.sin(2 * np.pi * doy / 365))

add_base_features(df)

,Date,Day,Month,Year,Season,CHL,CHL_uncertainty,SST,UI,YDOY,spatial_cluster,HAB,complete,CHL_log,doy,sin_doy
0,2013-02-19,19,2,2013,Winter,NaN,NaN,12.63,0.003952,0.651899,0,0,False,NaN,50,0.758306
1,2013-02-27,27,2,2013,Winter,NaN,NaN,12.54,0.949088,0.541628,0,0,False,NaN,58,0.840618
2,2013-03-04,4,3,2013,Spring,NaN,NaN,13.56,0.147877,0.467359,0,0,False,NaN,63,0.884068
3,2013-03-13,13,3,2013,Spring,1.419613,19.60,12.60,1.041656,0.325342,0,0,True,0.350384,72,0.945596
4,2013-03-19,19,3,2013,Spring,NaN,NaN,12.26,0.003219,0.226116,0,0,False,NaN,78,0.974100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1058,2020-12-10,10,12,2020,Winter,NaN,NaN,13.99,-0.140240,0.941317,5,0,False,NaN,345,-0.337523
1059,2020-12-14,14,12,2020,Winter,NaN,NaN,13.21,-0.291187,0.962309,5,0,False,NaN,349,-0.271958
1060,2020-12-21,21,12,2020,Winter,NaN,NaN,13.72,-0.508855,0.988023,5,0,False,NaN,356,-0.154309
1061,2020-12-22,22,12,2020,Winter,2.599458,40.02,14.21,-0.237547,0.990532,4,0,True,0.955303,357,-0.137279


In [201]:
# ----------------------------- Imputação -----------------------------

def apply_strategy(D, strategy):
    """Aplica a estratégia de missing values a D = {"train": df, "val": df, "test": df}.
    Os valores de imputação são calculados apenas com D["train"].
    Devolve (D_novo, fills); fills guarda o que foi aprendido, para inspeção."""
    if strategy not in STRATEGIES:
        raise ValueError(f"Estratégia desconhecida: {strategy!r}. Opções: {STRATEGIES}")
    D = {k: d.copy() for k, d in D.items()}
    fills = {}

    if strategy == "drop_rows":                      # remover linhas com missing values
        D = {k: d.dropna(subset=MISS) for k, d in D.items()}

    elif strategy == "drop_features":                # remover as features com missing values
        D = {k: d.drop(columns=MISS) for k, d in D.items()}

    else:                                            # imputar
        key = {"season_median": "Season", "doy_window_median": "doy"}.get(strategy)   # None -> mediana global
        tr = D["train"]
        for col in MISS:
            obs = tr[tr[col].notna()]                # só valores observados no train
            g = obs[col].median()                    # mediana global (também serve de fallback)

            if strategy == "season_median":          # mediana por estação
                table = obs.groupby("Season")[col].median()
            elif strategy == "doy_window_median":    # mediana numa janela de ±DOY_HALF_WINDOW dias
                dist = np.abs(np.arange(1, 366)[:, None] - obs["doy"].to_numpy())
                dist = np.minimum(dist, 365 - dist)  # distância circular (31 dez ~ 1 jan)
                table = pd.Series([np.median(obs[col].to_numpy()[m]) if m.any() else g
                                   for m in dist <= DOY_HALF_WINDOW], index=range(1, 366))
            else:                                    # global_median
                table = g
            fills[col] = table

            for d in D.values():                     # aplicar a train, validation e test
                fill = d[key].map(table) if key else g
                d[col] = d[col].fillna(fill).fillna(g)

    return D, fills

In [202]:
# --------------------- Preprocessing (scaling) ---------------------

def preprocess(D):
    """standardização com média/desvio do train.
    Devolve (X, y), cada um um dicionário com as chaves de D."""
    cols = BASE_NUM + [c for c in MISS if c in D["train"].columns]    
    mu = D["train"][cols].mean()                                      # média e desvio só do train
    sd = D["train"][cols].std(ddof=0).replace(0, 1.0)

    X, y = {}, {}
    for k, d in D.items():
        X[k] = ((d[cols] - mu) / sd)
        y[k] = d["HAB"].astype(int)
    return X, y

## 3. Construir os datasets para cada estratégia

O **test não é usado aqui**: só `train` (para aprender) e `val_eval` (para avaliar). Quando fores avaliar o modelo final, aplica `transform(test, state)` com o `state` da estratégia escolhida.

In [203]:
def get_data(strategy):
    """Devolve (X_train, y_train, X_val, y_val, X_test, y_test) para a estratégia indicada."""
    base = {k: add_base_features(d) for k, d in splits.items()}       # 1. features base
    D, _ = apply_strategy(base, strategy)                             # 2. missing values (aprende no train)
    X, y = preprocess(D)                                              # 3. scaling (aprende no train)
    return X["train"], y["train"], X["val"], y["val"], X["test"], y["test"]

In [204]:
# Exemplo de uso
X_train, y_train, X_val, y_val, X_test, y_test = get_data("doy_window_median")
print(X_train.shape, X_val.shape, X_test.shape)

(744, 6) (161, 6) (87, 6)
